In [1]:
from pyspark.sql import SparkSession

import os
os.environ["PYSPARK_PYTHON"] = r"C:\Users\Administrator\AppData\Local\Programs\Python\Python310\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"C:\Users\Administrator\AppData\Local\Programs\Python\Python310\python.exe"

# If you're in a notebook, it's common to have an old SparkSession running.
# This makes the cell re-runnable safely.
spark = SparkSession.builder.master("local[2]").appName("pyspark-local").getOrCreate()
sc = spark.sparkContext

sc.setLogLevel("ERROR")

# data = [1, 2, 3, 4, 5]

# rdd = sc.parallelize(data, 2)
# print(rdd.collect())

# data2 = ["apple", "banana", "cherry", "date", "elderberry"]
# rdd  = sc.parallelize(data2, 2)
# rdd_map = rdd.map(lambda fruit: (fruit,1))
# print(rdd_map.collect())

# nums = sc.parallelize([1, 2, 3, 4, 5], 2)
# even = nums.filter(lambda x: x % 2 == 0).filter(lambda x: x > 2)
# print(even.collect())

# rdd1 = sc.parallelize([1, 2, 3, 4, 5])
# rdd2 = sc.parallelize([3, 4, 5,6,7])
# rdd3 = rdd1.union(rdd2)
# print(rdd3.collect())
# rdd4 = rdd1.intersection(rdd2)
# print(rdd4.collect())
# pairs = nums.map(lambda x: (x, x * 2))
# pairs.collect()

# nums = sc.parallelize([1, 2, 3, 4, 5], 2)
# pairs = nums.map(lambda x: (x, x * 2))
# pairs.collect()

# doubled_values = pairs.mapValues(lambda v: v * 2)
# print(doubled_values.collect())

# sales = sc.parallelize([("Alice", 100), ("Bob", 200), ("Alice",50), ("Bob",30), ("Charlie", 70)])
# total_sales = sales.reduceByKey(lambda x, y: max(x,y) )
# print(total_sales.collect())

# rdd_text = sc.textFile("C:\\Users\\test.txt")  # Make sure the file path is correct
# rdd_map_text_flat_map = rdd_text.flatMap(lambda words: words.split(" "))
# print("Flat map text:",rdd_map_text_flat_map.collect())
# rdd_map_text_count = rdd_map_text_flat_map.count()
# print("Flat map total word count:", rdd_map_text_count)
# total_sales = rdd_map_text_flat_map.map(lambda word: (word, 1)).reduceByKey(lambda x, y: x + y)
# print("Flat map word counts:", total_sales.collect())

# ratings = sc.parallelize([
#     ("Avengers", 8.5), ("Titanic", 7.8), ("Avengers", 9.0),
#     ("Inception", 9.2), ("Titanic", 8.0), ("Inception", 8.9)
# ])

# Complex process --simpler process is below
# print("Raw ratings:", ratings.collect())
# rdd_flatmap_rating_total = ratings.map(lambda x: (x[0], x[1])).reduceByKey(lambda x, y: x + y)

# print("Rating total average:", rdd_flatmap_rating_total.collect())
# rdd_flatmap_rating_cnt = ratings.map(lambda word: (word[0], 1)).reduceByKey(lambda x, y: x + y)

# print("Rating total count:", rdd_flatmap_rating_cnt.collect())
# avg_rating = rdd_flatmap_rating_total.join(rdd_flatmap_rating_cnt).mapValues(lambda x: x[0] / x[1])
# print("Average rating:", avg_rating.collect())


#simplified process -- please follow like this
# # (key) -> (sum, count)
# sum_count = ratings.map(lambda x: (x[0], (x[1], 1))).reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
# print("Sum count:", sum_count.collect())

# # average = sum / count
# avg_rating = sum_count.mapValues(lambda sc: sc[0] / sc[1])
# print("Average rating new:", avg_rating.collect())


# cache_test = sc.parallelize(["I am going to jungle and going to eat and enjoy."]*10000)
# rdd_map_text_flat_map = cache_test.flatMap(lambda words: words.split(" "))
# print(rdd_map_text_flat_map.distinct().count())
# print(rdd_map_text_flat_map.count())

#spark.stop()


In [11]:
cache_test = sc.parallelize(["I am going to jungle and going to eat and enjoy."]*100000)
rdd_map_text_flat_map = cache_test.flatMap(lambda words: words.split(" ")).distinct()
print(rdd_map_text_flat_map.count())

8


In [19]:
rdd_map_text_flat_map.cache() 

PythonRDD[28] at RDD at PythonRDD.scala:53

In [21]:
print(rdd_map_text_flat_map.count())

8


In [ ]:
rdd_map_text_flat_map.unpersist()

PythonRDD[20] at RDD at PythonRDD.scala:53

In [ ]:
broadcast_var = sc.broadcast("This is a broadcast variable example.") 
print(broadcast_var.value)

This is a broadcast variable example.


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("AccumulatorDemo").getOrCreate()
sc = spark.sparkContext

# Create counter (starts at 0)
bad_records = sc.accumulator(0)

# Sample messy data
data = ["valid:100", "invalid", "valid:200", "empty:", "valid:150"]
rdd = sc.parallelize(data)

# Count bad records across ALL nodes
def check_record(line):
    global bad_records
    if ":" not in line or line == "empty:":
        bad_records.add(1)  # Executors update counter!

rdd.foreach(check_record)

print(f"Bad records found: {bad_records.value}")  # 2




# words_cached = lines.flatMap(lambda line: line.split()).cache()

# start = time.time()

# count2 = words_cached.count()
# distinct2 = words_cached.distinct().count()

# print(f"WITH CACHE:")
# print(f"  Word count: {count2}")
# print(f"  Unique words: {distinct2}")
# print(f"  TIME: {time.time() - start:.2f} seconds")